# 📘 Module 06 — PyTorch from Scratch
## Chapter 6.4 — Datasets and DataLoaders: Building Scalable Data Pipelines

> **Learning Path:** Zero to Hero — AI Engineer  
> **Module:** 06 of 18  
> **Topics:** Decoupled Data Pipeline Architecture, Custom `Dataset` (`__len__`, `__getitem__`), `DataLoader` Batching, Shuffling & Workers, Image Transforms & Data Augmentation  
> **Format:** Notes first → Code second  

---


## 🎯 What You Will Learn

By the end of this chapter, you will be able to:

- [ ] Explain the producer-consumer design pattern decoupling data storage (`Dataset`) from batch streaming (`DataLoader`)
- [ ] Implement custom PyTorch `Dataset` classes supporting lazy loading and index-based access
- [ ] Configure `DataLoader` parameters: batch size, shuffling, parallel multi-processing workers, and memory pinning
- [ ] Build custom image transformations and data augmentation pipelines (normalization, random erasing, noise)
- [ ] Stream real MNIST handwritten digits directly from local workspace storage into PyTorch batches

**Estimated Reading Time:** ~30 minutes


## 🗂️ Table of Contents

1. [The Pipeline Architecture — Decoupling Data Storage from Model Execution](#1-the-pipeline-architecture--decoupling-data-storage-from-model-execution)
2. [Building Custom Datasets with `torch.utils.data.Dataset`](#2-building-custom-datasets-with-torchutilsdatadataset)
3. [The DataLoader Engine — Batching, Shuffling, & Multi-Worker Streaming](#3-the-dataloader-engine--batching-shuffling--multi-worker-streaming)
4. [Transforms & Data Augmentation — Composable Pipelines](#4-transforms--data-augmentation--composable-pipelines)
5. [Chapter Summary](#-chapter-summary)
6. [Resources & Further Reading](#-resources--further-reading)
7. [Next Chapter Preview](#-next-chapter-chapter-65--a-complete-training-loop-from-scratch)


---

## 1. The Pipeline Architecture — Decoupling Data Storage from Model Execution

---

### 🗣️ In Simple English

Imagine a Michelin-starred restaurant kitchen with a world-class Head Chef (the **GPU**):
- The Chef is lightning-fast: she can cook a steak in $10\text{ seconds}$.
- But what happens if the Chef has to personally run outside to the farm, dig up carrots, wash the dirt off, chop the onions, and carry them back into the kitchen before cooking each dish?
- The restaurant grinds to a halt! The expensive Chef spends $99\%$ of her time walking to the farm instead of cooking.

To fix this, the kitchen hires a **Prep Crew** (the **CPU DataLoaders**):
- While the Chef is cooking Order #1, the Prep Crew is already washing, chopping, and assembling the ingredients for Order #2, #3, and #4 in parallel.
- When the Chef finishes Order #1, Order #2 is already sitting on the counter, prepped and ready to go!

In PyTorch:
- **`Dataset`:** The storage pantry. It knows how many items exist, and how to fetch item #42.
- **`DataLoader`:** The multi-threaded prep crew. It reads data in the background, groups samples into batches, shuffles their order, and keeps the GPU fully fed with zero idle downtime!

---

### 🌍 Real-Life Example

**Scenario:** Training an autonomous driving perception network on $500\text{ GB}$ of multi-camera driving video.

| Pipeline Component | Role in Autonomous Driving Training | Hardware Utilized |
|:---|:---|:---|
| **Storage (`Dataset`)** | Reads video frames from SSD; decodes H.264 video to raw RGB tensors | High-speed NVMe Storage |
| **Transform** | Normalizes pixel values; applies random rain/fog simulation and camera jitter | Multi-Core CPU Threads |
| **Conveyor (`DataLoader`)** | Prefetches 64 images across 4 parallel background worker threads | Host RAM (Page-Locked Pin Memory) |
| **Model Engine** | Ingests pre-batched tensor `(64, 3, 720, 1280)` and computes forward/backward pass | GPU Tensor Cores ($100\%$ Saturation!) |

---

### 📖 How It Works (Step by Step)

1. **Separation of Concerns:**
   - Code that reads datasets from files, databases, or cloud buckets should **never be mixed with model forward-pass code**.
   - Wrapping storage in `torch.utils.data.Dataset` makes data access interchangeable. You can swap an MNIST dataset for a CIFAR dataset without changing a single line of model or training code!
2. **The Iteration Cycle:**
   ```
   Dataset[idx] ──► Transform(sample) ──► Collate into Batch ──► GPU Transfer ──► Model Forward
   ```

---

### 🔑 Key Concepts

| Class | Primary Responsibility | Key Methods |
|:---|:---|:---|
| **`Dataset`** | Abstracts a single training example and its ground-truth label. | `__len__()`, `__getitem__(idx)` |
| **`DataLoader`** | Combines a dataset with a sampler, multi-processing workers, and batch collator. | `__iter__()` (Yields batches) |
| **`Collate Function`** | Merges a list of individual samples `[(x1, y1), (x2, y2)]` into a batched tensor `(X_batch, Y_batch)`. | `collate_fn` |

---

### 🗺️ Visual Overview

```
THE PRODUCER-CONSUMER PARALLEL STREAMING ARCHITECTURE:

  CPU Disk Storage                 CPU Worker Threads (DataLoader)                 GPU VRAM (Model)
┌──────────────────┐             ┌──────────────────────────────────┐            ┌──────────────────┐
│  Dataset Pantry  │             │ Worker 1: Prepping Batch t+1     │            │  Model Training  │
│                  │ ──────────► │ Worker 2: Prepping Batch t+2     │ ─────────► │                  │
│ [Image files on  │             │ Worker 3: Prepping Batch t+3     │  (PCIe)    │ Computing Batch t│
│  NVMe SSD disk]  │             │ (Shuffling, Resizing, Norm)      │            │ (100% Saturated!)│
└──────────────────┘             └──────────────────────────────────┘            └──────────────────┘
```

---

### 🚫 Common Misconceptions

| ❌ Myth | ✅ Reality |
|:---|:---|
| *"I should convert my entire 50GB dataset into a gigantic PyTorch tensor before training."* | **A recipe for disaster!** Storing massive datasets directly as monolithic tensors will exhaust system RAM and cause immediate memory crashes. Professional pipelines load data **lazily on-demand** inside `__getitem__()`. |

---

### ✅ When to Use This

- Always wrap every dataset in a `Dataset` and stream it via a `DataLoader`.

---

### 📊 Section Quick Reference

| Class | Imported From | Key Arguments |
|:---|:---|:---|
| `Dataset` | `torch.utils.data.Dataset` | Subclassed by user |
| `DataLoader` | `torch.utils.data.DataLoader` | `dataset, batch_size, shuffle, num_workers` |

---


---

## 2. Building Custom Datasets with `torch.utils.data.Dataset`

---

### 🗣️ In Simple English

How do you build a custom dataset in PyTorch?

You create a Python class that inherits from `torch.utils.data.Dataset` and implement **exactly TWO magic methods**:
1. **`__len__(self)`:** Returns how many total items exist in the dataset (`len(dataset)`).
2. **`__getitem__(self, idx)`:** Takes an integer index `idx` (say, `42`) and returns the $42^{\text{nd}}$ data sample and its label as a tuple: `(feature_tensor, label_tensor)`.

That's it! As long as your class provides those two methods, PyTorch considers it a valid dataset and can immediately plug it into DataLoaders, parallel worker pools, and distributed training clusters.

---

### 📖 How It Works (Step by Step)

#### In-Memory Loading vs Lazy Loading
1. **In-Memory Datasets (Small Data):**
   - In `__init__()`, load all arrays directly into RAM.
   - In `__getitem__(idx)`, simply slice the array: `return self.X[idx], self.y[idx]`.
   - Perfect for MNIST, CIFAR-10, or tabular CSV files under a few gigabytes.
2. **Lazy Datasets (Large Data):**
   - In `__init__()`, only store a list of image filenames or paths: `self.image_paths = [...]`.
   - In `__getitem__(idx)`, read **only the single requested image from disk**, decode it into a tensor, apply transforms, and return it.
   - Allows training on multi-terabyte datasets that are $100\times$ larger than your physical computer RAM!

---

### 🔑 Key Concepts

| Magic Method | What It Does | Example Return Value |
|:---|:---|:---:|
| `def __len__(self)` | Answers: *"How big is this dataset?"* | `len(self.labels)` (e.g. `60000`) |
| `def __getitem__(self, idx)` | Answers: *"Give me sample number idx"* | `(image_tensor, label_tensor)` |
| `self.transform` | Optional callable function modifying the sample before returning. | Resizing, normalization |

---

### 🗺️ Visual Overview

```
THE DATASET PROTOCOL CONTRACT:

  class MyCustomDataset(Dataset):
      def __init__(self, data_path, transform=None):
          # Load data or file paths

      def __len__(self):
          return len(self.samples)          <─── Allows writing: `len(dataset)`

      def __getitem__(self, idx):
          sample, label = self.samples[idx]
          if self.transform:
              sample = self.transform(sample)
          return sample, label               <─── Allows writing: `dataset[42]`
```

---

### 🚫 Common Misconceptions

| ❌ Myth | ✅ Reality |
|:---|:---|
| *"Can `__getitem__()` return a dictionary instead of a tuple?"* | **Yes!** While `(x, y)` tuples are the traditional standard, `__getitem__()` can return dictionaries (e.g., `{'input_ids': ..., 'attention_mask': ..., 'labels': ...}`), which is the standard convention used by HuggingFace Transformers. |

---

### ✅ When to Use This

- Subclass `Dataset` for custom tabular CSVs, proprietary image folders, audio recordings, or text corpora.

---

### 📊 Section Quick Reference

| Method | Signature | Mandatory? |
|:---|:---|:---:|
| `__init__` | `def __init__(self, ...)` | ✅ Yes |
| `__len__` | `def __len__(self) -> int` | ✅ Yes |
| `__getitem__` | `def __getitem__(self, idx: int) -> tuple` | ✅ Yes |

---


In [1]:
# ✅ Code Demo: Building a Custom PyTorch Dataset Loading Real MNIST from Disk
# Concept: Subclassing Dataset, Implementing __len__ and __getitem__, and Tensor Conversion
# -----------------------------------------------------------------------------------------

import os
import torch
from torch.utils.data import Dataset
import numpy as np

# Ensure local asset path exists
assets_dir = 'assets' if os.path.exists('assets') else 'Module-06-PyTorch-Foundations/assets'
os.makedirs(assets_dir, exist_ok=True)

# 1. Custom MNIST Dataset Class
class LocalMNISTDataset(Dataset):
    def __init__(self, npz_path, train=True, transform=None):
        # Verify local file exists strictly in workspace
        if not os.path.exists(npz_path):
            raise FileNotFoundError(f"Dataset file not found at: {npz_path}")
            
        data = np.load(npz_path)
        if train:
            self.images = data['x_train']  # Shape: (60000, 28, 28)
            self.labels = data['y_train']  # Shape: (60000,)
        else:
            self.images = data['x_test']   # Shape: (10000, 28, 28)
            self.labels = data['y_test']   # Shape: (10000,)
            
        self.transform = transform

    def __len__(self):
        # Mandatory Protocol 1: Total sample count
        return len(self.labels)

    def __getitem__(self, idx):
        # Mandatory Protocol 2: Retrieve single sample at index idx
        img = self.images[idx]
        label = int(self.labels[idx])
        
        # Convert uint8 numpy array [0, 255] to float32 tensor normalized to [0, 1]
        # Add channel dimension: (28, 28) -> (1, 28, 28)
        img_tensor = torch.from_numpy(img).float().unsqueeze(0) / 255.0
        
        # Apply optional transforms
        if self.transform is not None:
            img_tensor = self.transform(img_tensor)
            
        label_tensor = torch.tensor(label, dtype=torch.long)
        return img_tensor, label_tensor

# 2. Instantiate Dataset from Local Workspace Path
mnist_path = 'data/mnist.npz' if os.path.exists('data/mnist.npz') else 'Module-06-PyTorch-Foundations/data/mnist.npz'
train_dataset = LocalMNISTDataset(npz_path=mnist_path, train=True)
test_dataset  = LocalMNISTDataset(npz_path=mnist_path, train=False)

print("=" * 65)
print("CUSTOM LOCAL MNIST DATASET INSPECTION")
print("=" * 65)
print(f"Training Dataset Length: {len(train_dataset):,} samples")
print(f"Testing Dataset Length:  {len(test_dataset):,} samples")
print("-" * 65)

# 3. Test Indexing into Single Sample (Verifying __getitem__)
sample_img, sample_lbl = train_dataset[42]
print(f"Sample #42 Image Shape: {sample_img.shape} [Channels=1, Height=28, Width=28]")
print(f"Sample #42 Image Dtype: {sample_img.dtype}")
print(f"Sample #42 Pixel Range: [{sample_img.min():.2f}, {sample_img.max():.2f}]")
print(f"Sample #42 Ground Truth Label: {sample_lbl.item()} (Digit: {sample_lbl.item()})")
print("=" * 65)


CUSTOM LOCAL MNIST DATASET INSPECTION
Training Dataset Length: 60,000 samples
Testing Dataset Length:  10,000 samples
-----------------------------------------------------------------
Sample #42 Image Shape: torch.Size([1, 28, 28]) [Channels=1, Height=28, Width=28]
Sample #42 Image Dtype: torch.float32
Sample #42 Pixel Range: [0.00, 1.00]
Sample #42 Ground Truth Label: 7 (Digit: 7)


---

## 3. The DataLoader Engine — Batching, Shuffling, & Multi-Worker Streaming

---

### 🗣️ In Simple English

Now that our `Dataset` can return single samples `dataset[i]`, how do we turn those individual samples into **batches of 64 or 128 tensors** ready for GPU training?

We wrap the dataset in a **`DataLoader`**.

The `DataLoader` takes care of all the messy engineering plumbing:
1. **Batching:** It calls `__getitem__` 64 times and stacks the images into a single 4D batch tensor: `(64, 1, 28, 28)`.
2. **Shuffling:** At the start of every epoch, it randomizes the index order so the model never memorizes the sequence.
3. **Multi-Processing (`num_workers`):** It spawns parallel worker processes that load the next batch while the GPU is busy training on the current batch!
4. **Memory Pinning (`pin_memory=True`):** It locks the batch in host RAM, enabling direct, high-speed DMA (Direct Memory Access) transfers across the PCIe bus to GPU VRAM!

---

### 📖 How It Works (Step by Step)

#### The Essential DataLoader Parameters

```python
from torch.utils.data import DataLoader

train_loader = DataLoader(
    dataset=train_dataset,
    batch_size=64,         # Number of samples per training step
    shuffle=True,          # Shuffle order every epoch (Crucial for training!)
    num_workers=2,         # 2 parallel CPU worker processes
    pin_memory=True,       # Fast PCIe transfers to GPU
    drop_last=True         # Discard incomplete final batch
)
```

#### Why `drop_last=True` Matters
If your dataset has $1,005$ samples and your batch size is $100$:
- Batches 1 through 10 have size $100$.
- Batch 11 has only $5$ samples!
- In layers like `BatchNorm2d`, running a tiny batch of 5 samples can cause statistical noise spikes or shape dimension mismatch crashes. Setting `drop_last=True` discards that final partial batch to keep all batches identical in size.

#### Training vs Testing DataLoader Rules
- **Training DataLoader:** `shuffle=True` (randomize every epoch to prevent overfitting).
- **Validation / Test DataLoader:** `shuffle=False` (we want deterministic, reproducible evaluation scores!).

---

### 🔑 Key Concepts

| Parameter | Standard Value | What It Does |
|:---|:---:|:---|
| `batch_size` | $32, 64, 128$ | Number of samples grouped per forward pass. |
| `shuffle` | `True` (Train) / `False` (Test) | Randomizes index access order. |
| `num_workers` | $2 \dots 8$ ($0$ on Windows/debug) | Number of parallel CPU subprocesses loading data. |
| `pin_memory` | `True` (when using CUDA) | Speeds up CPU $\to$ GPU memory copy speed. |

---

### 🗺️ Visual Overview

```
HOW DATALOADER TRANSFORMS SINGLE SAMPLES INTO BATCHES:

  Dataset:
    sample 0: (1, 28, 28) ───\
    sample 1: (1, 28, 28) ────\    DataLoader Collation
    sample 2: (1, 28, 28) ─────► [ torch.stack() ] ───► Batch X: (64, 1, 28, 28)
    ...                       /                           Batch y: (64,)
    sample 63: (1, 28, 28) ──/
```

---

### 🚫 Common Misconceptions

| ❌ Myth | ✅ Reality |
|:---|:---|
| *"Setting `num_workers=16` is always $16\times$ faster than `num_workers=2`."* | **False!** Spawning too many worker processes creates severe CPU thread contention, IPC (Inter-Process Communication) overhead, and memory thrashing. A rule of thumb is setting `num_workers = min(4, os.cpu_count())`. (Note: On Windows, `num_workers=0` is often preferred in simple scripts to avoid multiprocessing pickle gotchas). |

---

### ✅ When to Use This

- Always stream data through `DataLoader`. Never write manual slicing loops!

---

### 📊 Section Quick Reference

| Mode | `shuffle` | `drop_last` | Purpose |
|:---|:---:|:---:|:---|
| **Training** | `True` | `True` | Stable gradient descent updates |
| **Validation / Test** | `False` | `False` | Exact, deterministic score calculation |

---


In [2]:
# ✅ Code Demo: DataLoader Batching, Shuffling, and Batch Iteration
# Concept: Batch Formation, Iteration Protocols, and Inspecting Batch Dimensions
# -----------------------------------------------------------------------------

import os
import torch
from torch.utils.data import DataLoader

# Ensure local asset path exists
assets_dir = 'assets' if os.path.exists('assets') else 'Module-06-PyTorch-Foundations/assets'
os.makedirs(assets_dir, exist_ok=True)

# 1. Create Training and Testing DataLoaders
# Using batch_size=64
train_loader = DataLoader(
    dataset=train_dataset,
    batch_size=64,
    shuffle=True,          # Shuffle for training!
    num_workers=0,         # 0 for safe cross-platform execution
    drop_last=True
)

test_loader = DataLoader(
    dataset=test_dataset,
    batch_size=128,
    shuffle=False,         # Deterministic for testing!
    num_workers=0,
    drop_last=False
)

print("=" * 65)
print("PYTORCH DATALOADER BATCHING INSPECTION")
print("=" * 65)
print(f"Total Training Samples: {len(train_dataset):,}")
print(f"Batch Size:             64")
print(f"Total Batches per Epoch: {len(train_loader):,} batches ({len(train_dataset)} // 64)")
print(f"Total Testing Batches:  {len(test_loader):,} batches (batch_size=128)")
print("-" * 65)

# 2. Iterate through First Batch
# Calling next(iter(...)) simulates 1 step of training loop
batch_images, batch_labels = next(iter(train_loader))

print("FIRST BATCH TENSOR DIMENSIONS:")
print(f"  Batch Images Shape: {batch_images.shape} (Batch=64, Channel=1, Height=28, Width=28)")
print(f"  Batch Labels Shape: {batch_labels.shape} (Batch=64 labels)")
print(f"  First 10 Labels:    {batch_labels[:10].tolist()}")
print("=" * 65)


PYTORCH DATALOADER BATCHING INSPECTION
Total Training Samples: 60,000
Batch Size:             64
Total Batches per Epoch: 937 batches (60000 // 64)
Total Testing Batches:  79 batches (batch_size=128)
-----------------------------------------------------------------
FIRST BATCH TENSOR DIMENSIONS:
  Batch Images Shape: torch.Size([64, 1, 28, 28]) (Batch=64, Channel=1, Height=28, Width=28)
  Batch Labels Shape: torch.Size([64]) (Batch=64 labels)
  First 10 Labels:    [0, 2, 9, 2, 8, 9, 9, 1, 4, 9]


---

## 4. Transforms & Data Augmentation — Composable Pipelines

---

### 🗣️ In Simple English

A neural network trained only on pristine, perfectly centered handwritten digits will fail when deployed in the real world on a messy phone camera. If a user writes a digit slightly tilted, off-center, or with poor lighting, an un-augmented model gets confused!

**Data Augmentation** solves this by artificially creating infinite variations of your training images on the fly:
- Tilted digits (Random Rotation)
- Off-center digits (Random Translation / Affine Shifts)
- Thicker or thinner strokes (Random Scaling / Noise)

The beauty of transforms is that **no new images are saved to your hard drive**. Every time a worker process grabs an image from disk, it applies a random nudge. The network never sees the exact same image twice!

---

### 📖 How It Works (Step by Step)

#### Building a Composable Transform Pipeline
A transform is simply any callable object (a function or class with `__call__`) that takes an input tensor and returns a transformed tensor.

We can chain multiple transforms together into a single pipeline:
```python
class ComposedTransforms:
    def __init__(self, transforms):
        self.transforms = transforms
        
    def __call__(self, x):
        for t in self.transforms:
            x = t(x)
        return x
```

#### Standard Normalization Formula
To accelerate gradient descent, we normalize image channels using dataset-wide mean ($\mu$) and standard deviation ($\sigma$):
$$x_{\text{norm}} = \frac{x - \mu}{\sigma}$$
For MNIST (which has 1 channel):
- Mean $\mu \approx 0.1307$
- Std $\sigma \approx 0.3081$
After normalization, the background pixels sit around $-0.42$, and the white stroke pixels sit around $+2.82$, centered around zero!

---

### 🔑 Key Concepts

| Transformation | Real-World Invariance Learned |
|:---|:---|
| **Normalization ($(x - \mu)/\sigma$)** | Zero-centers inputs; stabilizes initial layer weight updates. |
| **Random Shift / Translation** | Translation invariance (digit doesn't have to be perfectly centered). |
| **Random Gaussian Noise** | Sensor noise robustness (prevents overfitting to camera artifacts). |
| **Random Rotation** | Orientation invariance (handles human writing handwriting slant). |

---

### 🗺️ Visual Overview

```
THE ON-THE-FLY DATA AUGMENTATION PIPELINE:

  Raw Stored Image               Transform Pipeline                          Augmented Tensor
┌──────────────────┐           ┌────────────────────────────────┐          ┌──────────────────┐
│ Digit '7' on SSD │ ────────► │ 1. Random Shift (-2px, +3px)   │ ───────► │ Digit '7' tilted │
│                  │           │ 2. Add Gaussian Noise          │          │ & normalized     │
│ [Exact same file]│           │ 3. Normalize (x - mu) / sigma  │          │ (Unique sample!) │
└──────────────────┘           └────────────────────────────────┘          └──────────────────┘
```

---

### 🚫 Common Misconceptions

| ❌ Myth | ✅ Reality |
|:---|:---|
| *"Should I apply random rotation and random flips to my test/validation set?"* | **NEVER augment test data!** Data augmentation is strictly a **training regularizer**. Test and validation sets must remain pure, un-augmented ground truth to accurately measure real-world performance. |

---

### ✅ When to Use This

- Always normalize inputs to zero-mean and unit variance.
- Apply mild spatial augmentations to computer vision training datasets.

---

### 📊 Section Quick Reference

| Transform | Apply to Train? | Apply to Test? |
|:---|:---:|:---:|
| **Normalization ($(x-\mu)/\sigma$)** | ✅ Yes | ✅ Yes (using training stats) |
| **Random Jitter / Affine Shift** | ✅ Yes | ❌ No |
| **Random Flip / Noise** | ✅ Yes | ❌ No |

---


In [3]:
# ✅ Code Demo: Pure PyTorch Image Transforms & Data Augmentation Visualizer
# Concept: Custom Composable Transforms, Random Shifts, Normalization, and Grid Plotting
# --------------------------------------------------------------------------------------

import os
import torch
import numpy as np
import matplotlib.pyplot as plt

# Ensure local asset path exists
assets_dir = 'assets' if os.path.exists('assets') else 'Module-06-PyTorch-Foundations/assets'
os.makedirs(assets_dir, exist_ok=True)

# 1. Modular Transform Classes in Pure PyTorch
class NormalizeTensor:
    def __init__(self, mean=0.1307, std=0.3081):
        self.mean = mean
        self.std = std
    def __call__(self, tensor):
        return (tensor - self.mean) / self.std

class RandomShift:
    def __init__(self, max_shift=3):
        self.max_shift = max_shift
    def __call__(self, tensor):
        # tensor shape: (1, 28, 28)
        shift_x = np.random.randint(-self.max_shift, self.max_shift + 1)
        shift_y = np.random.randint(-self.max_shift, self.max_shift + 1)
        # Roll tensor along spatial axes
        shifted = torch.roll(tensor, shifts=(shift_y, shift_x), dims=(1, 2))
        return shifted

class AddGaussianNoise:
    def __init__(self, std=0.08):
        self.std = std
    def __call__(self, tensor):
        noise = torch.randn_like(tensor) * self.std
        return torch.clamp(tensor + noise, 0.0, 1.0)

# 2. Composable Transform Pipeline
class ComposePipeline:
    def __init__(self, transforms):
        self.transforms = transforms
    def __call__(self, x):
        for t in self.transforms:
            x = t(x)
        return x

# Build augmentation pipeline
augmentation_pipeline = ComposePipeline([
    RandomShift(max_shift=3),
    AddGaussianNoise(std=0.08)
])

# 3. Visualizing Original vs Augmented Images
# Fetch single raw sample digit
raw_img, raw_lbl = train_dataset[7]  # Digit sample

fig, axes = plt.subplots(1, 5, figsize=(13, 3))

# Panel 0: Original Un-augmented Image
axes[0].imshow(raw_img.squeeze(), cmap='gray')
axes[0].set_title(f'Original Raw\n(Digit: {raw_lbl.item()})', fontsize=10, fontweight='bold', color='#1e293b')
axes[0].axis('off')

# Panels 1-4: 4 Stochastic Augmented Variations of the exact same image!
np.random.seed(42)
for i in range(1, 5):
    augmented_sample = augmentation_pipeline(raw_img.clone())
    axes[i].imshow(augmented_sample.squeeze(), cmap='gray')
    axes[i].set_title(f'Augmented #{i}\n(Shift + Noise)', fontsize=10, fontweight='bold', color='#2563eb')
    axes[i].axis('off')

plt.suptitle('On-The-Fly Data Augmentation: 1 Image -> Infinite Training Variations', fontsize=12, fontweight='bold')
plt.tight_layout()

fig_path = os.path.join(assets_dir, 'mnist_augmentation_examples.png')
plt.savefig(fig_path, dpi=120)
plt.close()
print(f"Figure successfully saved to: {fig_path}")

# 4. Normalization Numerical Verification
norm_transform = NormalizeTensor(mean=0.1307, std=0.3081)
normed_img = norm_transform(raw_img)

print("=" * 65)
print("NORMALIZATION NUMERICAL VERIFICATION")
print("=" * 65)
print(f"Raw Image Range:        [{raw_img.min():.4f}, {raw_img.max():.4f}] (Mean={raw_img.mean():.4f})")
print(f"Normalized Image Range: [{normed_img.min():.4f}, {normed_img.max():.4f}] (Zero-centered around 0)")
print("=" * 65)


Figure successfully saved to: Module-06-PyTorch-Foundations/assets\mnist_augmentation_examples.png
NORMALIZATION NUMERICAL VERIFICATION
Raw Image Range:        [0.0000, 1.0000] (Mean=0.1794)
Normalized Image Range: [-0.4242, 2.8215] (Zero-centered around 0)


---

## 📝 Chapter Summary

| Component | Role | Mandatory Methods / Arguments | Crucial Best Practice |
|:---|:---|:---|:---|
| **`Dataset`** | Storage abstraction for single samples | `__len__()`, `__getitem__(idx)` | Load massive data lazily inside `__getitem__` to avoid RAM crashes. |
| **`DataLoader`** | Parallel streaming and batching engine | `dataset, batch_size, shuffle, num_workers` | Set `shuffle=True` for training; `shuffle=False` for testing. |
| **`pin_memory`** | Page-locked host memory allocation | `pin_memory=True` | Speeds up DMA tensor transfers across PCIe bus to GPU VRAM. |
| **`drop_last`** | Drops incomplete trailing batch | `drop_last=True` | Prevents batch-size mismatch crashes in training loops. |
| **Transforms** | Image normalization & augmentation | Callable `__call__(tensor)` | Never augment validation or test sets. |

---


## 🔗 Resources & Further Reading

| Resource | Type | Why Read It |
|:---|:---|:---|
| **[PyTorch Datasets & DataLoaders Official Tutorial](https://pytorch.org/tutorials/beginner/basics/data_tutorial.html)** | Official Guide | Clear guide on decoupling dataset code from model training. |
| **[PyTorch DataLoader Source Code Analysis](https://pytorch.org/docs/stable/data.html)** | Deep Architecture | Technical specification of multi-process data loading, worker seeding, and custom collate functions. |
| **[Fast Data Loading in PyTorch (Horace He)](https://chillee.github.io/programming/2020/05/30/faster-pytorch-data-loading.html)** | Performance Engineering | Profiling bottlenecks, memory pinning, and optimizing `num_workers`. |

---


## ➡️ Next Chapter: Chapter 6.5 — A Complete Training Loop from Scratch

Now that we have Tensors (6.1), Autograd (6.2), Neural Networks (6.3), and DataLoaders (6.4), we bring them all together into the canonical, production-grade PyTorch training pipeline:
- **The Canonical Heartbeat:** Forward pass $\to$ Loss $\to$ `zero_grad()` $\to$ `backward()` $\to$ `optimizer.step()`.
- **Training vs Validation Modes:** Toggling `model.train()` and `model.eval()`.
- **Metric Tracking:** Loss curves, running accuracies, and epoch timers.
- **Model Checkpoints:** Saving and restoring model weights using `torch.save()` and `state_dict`.

---
*Made with ❤️ | Zero to Hero — AI Engineer Learning Path*
